<a href="https://colab.research.google.com/github/omran35/MSIS822-Project/blob/main/AI_Arabic_Text_Detection_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from importlib.metadata import version, PackageNotFoundError

packages = ["datasets", "pandas", "numpy", "pyarrow", "notebook", "ipykernel"]

for package in packages:
    try:
        print(f"{package}=={version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed in this environment")

datasets==3.1.0
pandas==2.2.3
numpy==2.1.3
pyarrow==23.0.1
notebook==6.5.7
ipykernel==6.17.1


In [13]:
column_types = {}

for name, subset in dataset.items():
    column_types[name] = {
        column: feature.dtype
        for column, feature in subset.features.items()
    }

pd.DataFrame(column_types)

,by_polishing,from_title,from_title_and_content
original_abstract,string,string,string
allam_generated_abstract,string,string,string
jais_generated_abstract,string,string,string
llama_generated_abstract,string,string,string
openai_generated_abstract,string,string,string


In [12]:
all_data["group_id"] = pd.factorize(human_texts, sort=True)[0]

texts = all_data.melt(
    id_vars=["group_id", "generation_method"],
    value_vars=dataset["by_polishing"].column_names,
    var_name="text_source",
    value_name="text"
)

texts["label"] = (
    texts["text_source"] != "original_abstract"
).astype(int)

counts = texts["label"].value_counts().sort_index()

print("Human (0):", counts[0])
print("AI-generated (1):", counts[1])
print("Total text entries:", len(texts))

Human (0): 8388
AI-generated (1): 33552
Total text entries: 41940


In [11]:
frames = []

for name, subset in dataset.items():
    part = subset.to_pandas()
    part["generation_method"] = name
    frames.append(part)

all_data = pd.concat(frames, ignore_index=True)

human_texts = all_data["original_abstract"].str.strip()

print("Total rows:", len(all_data))
print("Unique human abstracts:", human_texts.nunique())
print("Repeated human entries:", human_texts.duplicated().sum())

Total rows: 8388
Unique human abstracts: 2992
Repeated human entries: 5396


In [10]:
import pandas as pd

checks = []

for name, subset in dataset.items():
    df = subset.to_pandas()

    checks.append({
        "subset": name,
        "rows": len(df),
        "missing_texts": df.isna().sum().sum(),
        "blank_texts": df.apply(
            lambda col: col.dropna().str.strip().eq("").sum()
        ).sum(),
        "duplicate_rows": df.duplicated().sum()
    })

quality_report = pd.DataFrame(checks)
quality_report

,subset,rows,missing_texts,blank_texts,duplicate_rows
0,by_polishing,2851,0,0,0
1,from_title,2963,0,0,0
2,from_title_and_content,2574,0,0,0


In [9]:
sample = dataset["by_polishing"][0]

print("Human-written abstract:")
print(sample["original_abstract"])

print("\nAI-generated abstract (OpenAI):")
print(sample["openai_generated_abstract"])

Human-written abstract:
كثيرا ما ارتبطت المصادر التاريخية في الأندلس خاصة منها كتب التراجم والفهرسات والبرامج وغيرها بدراسة حياة العلماء والرواة والقضاة والساسة ؛ وقد تطورت هذه المادة حتى ترك لنا المؤلفون الأندلسيون سلسلة متواصلة الحلقات من كتب التـراجم كالصلة لابن بشكوال ، وصلة الصلة لابن الزبير، والتكملة لكتاب الصلة لابن الآبار، والذيل والتكملة لكتابي الموصول والصلة لابن عبد الملك المراكشي إضافة إلى الإحاطة في أخبار غرناطة لابن الخطيب ، إلا أنها لم تنس أن تشير في ثنايا أو بالأحرى في خواتم هذه المؤلفات إلى فئة المرأة العالمة التي ساهمت في الإنتاج الفكري والحضاري الأندلسي. ومن خلالها سنسعى إلى الوقوف على حالة التعليم عند المرأة الأندلسية ، وكيف كانت تأخذ فنون العلم. وما مدى إسهامها في الفكر التربوي والإنتاج الفكري الأندلسيين ؟.

AI-generated abstract (OpenAI):
صور نظام التعليم عند المرأة الأندلسية تستند إلى دراسة دقيقة للمصادر التاريخية التي وثقت حياة العلماء والمثقفين في الأندلس، لا سيما كتب التراجم الشهيرة مثل "الصلة" لابن بشكوال و"صلة الصلة" لابن الزبير و"التكملة" لابن الآبار، وغيره

In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "KFUPM-JRCAI/arabic-generated-abstracts"
)

print(dataset)

DatasetDict({
    by_polishing: Dataset({
        features: ['original_abstract', 'allam_generated_abstract', 'jais_generated_abstract', 'llama_generated_abstract', 'openai_generated_abstract'],
        num_rows: 2851
    })
    from_title: Dataset({
        features: ['original_abstract', 'allam_generated_abstract', 'jais_generated_abstract', 'llama_generated_abstract', 'openai_generated_abstract'],
        num_rows: 2963
    })
    from_title_and_content: Dataset({
        features: ['original_abstract', 'allam_generated_abstract', 'jais_generated_abstract', 'llama_generated_abstract', 'openai_generated_abstract'],
        num_rows: 2574
    })
})


In [7]:
pip install "datasets==3.1.0"